In [30]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import nbformat
from IPython.display import HTML

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/Users/sureshkumar/UE WORKS/data-viz-class-material-w2026/data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  
Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and

In [31]:
region_avg = (
    df.groupby('Region')['Happiness_Score']
    .mean()
    .reset_index()
    .sort_values('Happiness_Score', ascending=True)
)

# Step 2: Build the chart strictly following all 4 rules
fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',                  # Rule 4: Horizontal bars for long category names
    text='Happiness_Score',
    # Rule 2: Insight title answering what stands out and why it matters
    title="<b>Western Europe Outpaces All Regions in Well-Being</b><br><sup>Regional average happiness scores range from 4.2 to 7.1 out of 10</sup>",
    labels={
        'Happiness_Score': 'Average Happiness Score (0–10 Scale)',
        'Region': ''                  # Clean look: remove redundant y-axis title
    },
    template='simple_white'           # Rule 3: Professional, minimalist background
)

# Styling and typography adjustments
fig.update_traces(
    marker_color='#1f77b4',           # Clean, professional single-hue color
    texttemplate='%{text:.2f}',       # Format score to 2 decimal places
    textposition='outside',           # Keep values readable and outside the bars
    cliponaxis=False
)

fig.update_layout(
    # Rule 1: Zero baseline strictly enforced on the quantitative axis
    xaxis=dict(
        range=[0, 8.5],
        zeroline=True,
        zerolinecolor='#333333',
        zerolinewidth=1.5,
        showgrid=True,
        gridcolor='#eaeaea'
    ),
    yaxis=dict(
        showline=False,
        ticks=''
    ),
    font=dict(
        family="Arial, sans-serif",
        size=12,
        color="#222222"
    ),
    title_font=dict(size=16),
    margin=dict(l=40, r=40, t=70, b=50),
    height=480
)

# Rule 3: Render directly inside the notebook cell without nbformat dependency issues
HTML(fig.to_html(include_plotlyjs='cdn'))

In [32]:
# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'

bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

# Sorting ascending ensures top countries appear at the top in a horizontal bar chart
combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score', ascending=True)
global_avg = df['Happiness_Score'].mean()

# Step 2: Build the contrast chart
fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    color='Group',
    orientation='h',
    text='Happiness_Score',
    # Rule 2: Insight title highlighting the deep disparity
    title="<b>A Divide of Over 5 Points Separates the World's Happiest and Most Vulnerable Nations</b><br><sup>Top 8 nations maintain scores above 7.0, while the bottom 8 fall below 3.5 — well under the global average</sup>",
    labels={'Happiness_Score': 'Happiness Score (0–10 Scale)', 'Country': ''},
    # Purposeful two-color contrast palette (calm teal vs. subdued coral)
    color_discrete_map={
        'Top 8': '#2b7bba',
        'Bottom 8': '#d95f02'
    },
    template='simple_white'
)

# Refined bar values and layout formatting
fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside',
    cliponaxis=False
)

# Rule 1: Zero baseline on x-axis + clutter reduction
fig.update_layout(
    xaxis=dict(
        range=[0, 8.8],
        zeroline=True,
        zerolinecolor='#333333',
        zerolinewidth=1.5,
        showgrid=True,
        gridcolor='#f0f0f0'
    ),
    yaxis=dict(
        showline=False,
        ticks=''
    ),
    legend=dict(
        title=None,
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    font=dict(family="Arial, sans-serif", size=12, color="#222222"),
    height=600,
    margin=dict(l=40, r=40, t=90, b=40)
)

# Stretch Goal: Vertical reference line showing the global average
fig.add_vline(
    x=global_avg,
    line_dash="dash",
    line_color="#7f7f7f",
    line_width=1.5,
    annotation_text=f"Global Avg: {global_avg:.2f}",
    annotation_position="top",
    annotation_font=dict(size=11, color="#555555")
)

# Visual gap annotation emphasizing the divide between the two groups
fig.add_annotation(
    x=global_avg,
    y=7.5,
    text="← Gap of > 3.5 points →",
    showarrow=False,
    font=dict(size=11, color="#777777", style="italic"),
    bgcolor="rgba(255, 255, 255, 0.85)",
    bordercolor="#e0e0e0",
    borderwidth=1,
    borderpad=4
)

# Render cleanly inside Jupyter notebook
HTML(fig.to_html(include_plotlyjs='cdn'))

In [33]:
# Step 1: Filter regions and compute regional averages for both sub-factors
target_regions = [
    'Western Europe',
    'Latin America',
    'East Asia',
    'Sub-Saharan Africa',
    'South Asia'
]

# Reshape into long (tidy) format suited for grouped bar charts in Plotly
grouped_data = (
    df[df['Region'].isin(target_regions)]
    .groupby('Region')[['GDP_per_capita', 'Freedom']]
    .mean()
    .reset_index()
    .melt(
        id_vars='Region',
        value_vars=['GDP_per_capita', 'Freedom'],
        var_name='Factor',
        value_name='Average_Score'
    )
)

# Step 2: Build the grouped bar chart
fig = px.bar(
    grouped_data,
    x='Region',
    y='Average_Score',
    color='Factor',
    barmode='group',
    text='Average_Score',
    # Rule 2: Insight title answering what stands out and why it matters
    title="<b>Economic Output Varies Widely Across Regions While Perceptions of Freedom Remain Narrower</b><br><sup>Sub-Saharan Africa and South Asia trail significantly in GDP contribution despite closer personal freedom parity</sup>",
    labels={
        'Region': '',
        'Average_Score': 'Average Score / Contribution',
        'Factor': 'Indicator'
    },
    # Meaningful contrast palette: dark slate blue for economic capital, warm amber for civil freedom
    color_discrete_map={
        'GDP_per_capita': '#1d4ed8',
        'Freedom': '#f59e0b'
    },
    template='simple_white'
)

# Refined data labels rounded to 2 decimal places
fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside',
    cliponaxis=False
)

# Rule 1: Zero baseline strictly enforced on the vertical y-axis
fig.update_layout(
    yaxis=dict(
        range=[0, grouped_data['Average_Score'].max() * 1.2],
        zeroline=True,
        zerolinecolor='#333333',
        zerolinewidth=1.5,
        showgrid=True,
        gridcolor='#f0f0f0'
    ),
    xaxis=dict(
        showline=False,
        ticks=''
    ),
    legend=dict(
        title=None,
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    font=dict(family="Arial, sans-serif", size=12, color="#222222"),
    margin=dict(l=40, r=40, t=90, b=40),
    height=500
)

# Render cleanly inside Jupyter notebook
HTML(fig.to_html(include_plotlyjs='cdn'))

KeyError: "Columns not found: 'GDP_per_capita'"